# NIT-4 · 4 — Pooling the sites

**The failure this notebook prevents.** Thirty-three plan arms across four
experiments, each reporting how much less its families worked than the controls.
Pool them and the family mean comes back with a standard error of half a percentage
point. It is a good number. It is also wrong three times over: it counted thirty-three
independent draws where there were four contractors; it averaged a five-year response
with three-year ones; and one of the four contractors changed what it meant by hours
half-way through and nobody in the pool was told.

`axiom.meta` pools the readouts notebook 3 filed, and this notebook is about the three
questions it asks before it does — *how many independent things are here*, *are these
the same quantity*, and *does every party mean the same thing by the outcome* — and about
the fourth question a programme asks after: of the decisions the four experiments
support, how many survive being decided together.

| | reaches into |
|---|---|
| the records | `meta.StudyRecord`, `meta.Corpus`, `meta.record_from_summary` |
| the classical pools | `meta.fixed_effect`, `meta.random_effects`, `meta.heterogeneity`, `meta.leave_one_out`, `meta.egger` |
| the party level | `meta.three_level`, `meta.pool` with `effect_key="nested"`, `meta.PoolPriors` |
| are these the same quantity | `meta.commensurable`, `Estimand.transfer_to`, `core.LatentSelection` |
| does every party mean the same thing | `io.DefinitionRegistry.consensus`, `commensurable(definitions=...)` |
| the decisions, taken together | `design.Readout`, `design.program_decisions`, `design.from_p_value` |
| the prior for the fifth site | `meta.prior_from_pool` |

In [ ]:
import sys

sys.path.insert(0, ".")

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import guarantee as W
from axiom.core import Intervention, LatentSelection, Population, Unsupported
from axiom.design import Readout, from_p_value, program_decisions
from axiom.identify import intention_to_treat
from axiom.io import Catalog, DefinitionRegistry, Program
from axiom.meta import (
    Commensurability, Corpus, Pooled, PoolPriors, PoolSpec, ThreeLevel, commensurable, egger,
    fixed_effect, leave_one_out, pool, prior_from_pool, random_effects, record_from_summary,
    three_level,
)
from axiom.viz import forest, funnel
from axiom.display import enable, show, table

from _style import caption, compare, dumbbell, intervals, lines, mark_x, mark_y, points

enable();  # every axiom result renders itself from here on

programme = W.run_programme()

## 1 · The records

One record per plan arm per enrolment cohort: the arm's mean hours change against the
site's control mean, among families still answering, as a **relative change** — hours
change over the site's pre-enrolment hours — so that the quantity is dimensionless and
poolable across sites whose families worked different amounts to begin with. The plan
each record ran under travels with it as a moderator, and so does the cohort's length.

In [ ]:
records, estimands, party_of = [], {}, {}
for site in programme.sites:
    run = programme[site.name]
    baseline = float(run.families.pre_hours.mean())
    for years in ([3] if len(site.durations) == 1 else [3, 5]):
        cohort = run.cohort(years)
        control = cohort.loc[cohort.assigned == 0, "hours_observed"]
        for arm in site.arms[1:]:
            treated = cohort.loc[cohort.arm == arm, "hours_observed"]
            g, t = site.plan_of(arm)
            study = f"{site.experiment}/{arm}/{years}y"
            records.append(record_from_summary(
                study=study, contributor=site.party, quantity="relative_change",
                estimate=float((treated.mean() - control.mean()) / baseline),
                se=float(np.sqrt(treated.var(ddof=1) / len(treated) + control.var(ddof=1) / len(control)) / baseline),
                read="experiment", family="negative_income_tax", n=int(len(treated) + len(control)),
                moderators={"guarantee": g, "tax_rate": t, "years": float(years)},
                period=str(site.started), source=site.experiment,
            ))
            estimands[study] = W.national_estimand(
                plan=(g, t), years=years, name=f"plan_{arm}_{years}y",
                reference_version="afdc_up_available" if site.afdc_up else "assigned",
            )
    party_of[site.party] = Program(party=site.party, program="income-maintenance")
corpus = Corpus(records=tuple(records), name="NIT-4")
y = np.array([r.estimate for r in corpus.records])
se = np.array([r.se for r in corpus.records])
print(f"{len(corpus)} records from {len(corpus.contributors())} parties; relative change from "
      f"{y.min():.1%} to {y.max():.1%}, median se {np.median(se):.1%}")
table([[r.study, f"{r.estimate:+.1%}", f"{r.se:.1%}", r.n] for r in corpus.records[:6]] + [["…", "", "", ""]],
      headers=("record", "relative change", "se", "families"), title="the first six")

In [ ]:
frame = corpus.to_frame()
labels = [r.study.replace("-1971", "").replace("-1970", "").replace("-1968", "") for r in corpus.records]
fig = intervals(
    [(lab, float(r.estimate), float(r.estimate - 1.96 * r.se), float(r.estimate + 1.96 * r.se))
     for lab, r in zip(labels, corpus.records)],
    ref=0.0, ref_label="no effect",
    title="Thirty-three plan arms, four contractors",
    subtitle="relative change in annual hours, plan arm against the site's control, with 95% intervals",
    x_title="relative change in hours", height=760,
)
caption(fig, "Every arm reduced hours and the reductions are not one number: Seattle-Denver's "
             "generous plans and five-year cohort sit far to the left, the Rural experiment's "
             "families responded more than Gary's at the same plans. Some of that is the plan — "
             "a moderator — and some of it is the party, which is the part a flat pool gets wrong.")

## 2 · The classical pools, and what they claim

Fixed effect first, because it is what a meeting does; then random effects, because
the heterogeneity statistic says the fixed effect is not entitled to its interval.

In [ ]:
fe = fixed_effect(y, se)
re = random_effects(y, se, tau_method="reml")
table([["fixed effect", f"{fe.estimate:+.2%}", fe.interval.text(), f"{fe.interval.width:.2%}", "—"],
       ["random effects (REML)", f"{re.estimate:+.2%}", re.interval.text(), f"{re.interval.width:.2%}", f"{re.tau:.2%}"]],
      headers=("pool", "mean", "95% interval", "width", "tau"))
h = re.heterogeneity
print(f"\nCochran's Q = {h.q:.1f} on {h.df} df (p = {h.p_value:.2g}); I² = {h.i2:.0%}: "
      f"{h.i2:.0%} of the spread between records is not sampling error")
loo = leave_one_out(y, se, method="dl")
worst = int(np.argmax(np.abs(loo.influence)))
print(f"leave-one-out: the most influential record is {corpus.records[worst].study}, "
      f"which moves the pooled mean by {loo.influence[worst]:.2f} pooled standard errors")
print(f"Egger's test for small-study asymmetry: p = {egger(y, se).p:.2f}")

## 3 · Thirty-three records, or four contractors?

Every record from one site was produced by one contractor, in one set of cities, with
one interviewing protocol, in one labour market. Four of Seattle-Denver's arms are not
four independent draws about the world. The random-effects pool above credits the
family mean with thirty-three of them.

`three_level` splits the spread into two variance components — between records within
a party, and between parties — and reports a family mean whose standard error reflects
the number of parties. `pool` with `effect_key="nested"` does the same thing as a
Bayesian tree, with the plan moderators in the mean, and shrinks each record toward
*its own party's* mean rather than toward the family's.

In [ ]:
party_index = [list(party_of).index(r.contributor) for r in corpus.records]
levels: ThreeLevel = three_level(y, se, list(range(len(corpus))), party_index)
rows = [["random effects, two-level", f"{re.estimate:+.2%}", re.interval.text(), f"{re.interval.width:.2%}", f"{re.tau:.2%}", "—"],
        ["three-level REML", f"{levels.estimate:+.2%}", levels.interval.text(), f"{levels.interval.width:.2%}",
         f"{levels.tau_study:.2%}", f"{levels.tau_party:.2%}"]]
fits = {}
for key, priors in (("study", PoolPriors(mu_scale=0.5, tau_scale=0.1)),
                    ("nested", PoolPriors(mu_scale=0.5, tau_scale=0.1, tau_party_fixed=max(levels.tau_party, 0.01)))):
    out = pool(PoolSpec(family="negative_income_tax", effect_key=key, priors=priors,
                        moderators=("guarantee", "tax_rate")), corpus, backend="laplace", draws=4000, seed=0)
    assert isinstance(out, Pooled)
    fits[key] = r = out.result
    rows.append([f"Bayesian, effect_key={key!r}, plan moderators", f"{r.mu.mean:+.2%}", r.mu.interval.text(),
                 f"{r.mu.interval.width:.2%}", f"{r.tau.mean:.2%}", "—" if r.tau_party is None else f"{r.tau_party.mean:.2%}"])
table(rows, headers=("pool", "mean", "interval", "width", "tau (within)", "tau (between parties)"),
      title="the same thirty-three records, four beliefs about what is independent")

In [ ]:
nested = fits["nested"]
table([[a.name.split("[")[1][:-1], f"{a.mean:+.2%}", a.interval.text(), f"{W.SITE_BY_NAME[next(s.name for s in W.SITES if s.party == a.name.split('[')[1][:-1])].site_scale:.2f}"]
       for a in nested.alphas],
      headers=("party", "party mean at the average plan", "interval", "the site's true scale (synthetic world)"),
      title="what the nested pool says about each contractor")
table([[g.name.split("_")[1], f"{g.mean:+.3f}", f"{g.sd:.3f}", g.interval.text()] for g in nested.gamma],
      headers=("moderator", "coefficient (relative change per unit)", "sd", "interval"),
      title="the plan, as a moderator of the relative change")
shrink = nested.shrinkage
fig = dumbbell(
    [s.effect.replace("-1971", "").replace("-1970", "").replace("-1968", "") for s in shrink[:12]],
    [s.estimate for s in shrink[:12]], [s.theta for s in shrink[:12]],
    before_label="what the arm reported", after_label="what the nested pool believes",
    title="Shrunk toward its own contractor, not toward the family",
    subtitle="the first twelve records under the nested pool; the target of each arrow is the party mean",
    x_title="relative change in hours",
)
caption(fig, "A noisy New Jersey arm is pulled toward New Jersey's mean, not toward a family "
             "mean that Seattle-Denver's four thousand families dominate. That is where the "
             "information about it actually is, and it is what the two-level pool cannot do.")

The two-level pools report the family mean to within two or three percentage points
and are not entitled to: they counted thirty-three markets where there were four
contractors. The three-level and nested intervals are eight points wide, and the
between-party component they estimate is larger than the within-party one — which is the
synthetic world's `site_scale` showing through, and the reason a fifth site's prior in
section 7 is wider than any of the four.

## 4 · Are these the same quantity?

Every record has an `Estimand` behind it, from notebook 3's filing. `commensurable`
transfers each one to a reference — the Rural arm at the plan Congress is discussing —
and reports what a pool over them would be assuming. Three things differ across the
corpus, and they are not equally bad.

In [ ]:
reference = next(r.study for r in corpus.records if r.study.startswith("RURAL") and "G0.75-t0.50" in r.study)
agreed: Commensurability = commensurable(corpus, estimands, reference=reference)
print(agreed.verdict().status, "—", agreed.verdict().reason)
table([[a.name, a.facet, a.statement[:90]] for a in sorted({a.name: a for a in agreed.assumptions}.values(), key=lambda a: a.name)],
      headers=("assumption", "facet", "what the pool would be assuming"),
      title=f"{len(agreed.checked)} records checked against {reference}")

`downgraded`, under named assumptions. A record at a different plan is pooled under
the assumption that the surface is correct between the two doses — which is exactly
what the moderators in section 3 are for. A five-year record is pooled under the
assumption that the response dynamics are stationary across windows — which the
synthetic world's `duration_factor` makes false, and which the `years` moderator would
have to absorb. And New Jersey's records, whose reference is *no plan with AFDC-UP
available*, are pooled under `version_irrelevance`: that a welfare alternative does not
change the contrast. Notebook 3 filed that deviation as `violated`.

None of those is a block, and that is the right call: a pool can average over a facet
difference an assumption licenses, as long as the assumption is on the result. Here is
the one it cannot average over.

In [ ]:
gary_arm = next(s for s in estimands if s.startswith("GARY"))
complier = estimands[gary_arm].model_copy(update={"population": Population(
    name="enrolled_families_below_150pct_poverty",
    latent=LatentSelection(kind="complier", instrument="assigned", exposure="enrolled", share=0.97))})
mixed = {**estimands, gary_arm: complier}
plan = complier.transfer_to(estimands[reference])
print("the transfer:", plan.status, "under", [a.name for a in plan.assumptions])
refused = commensurable(corpus, mixed, reference=reference)
print("the pool    :", refused.verdict().status)
entry = refused.entries[0]
table([[entry.study, entry.facet, entry.status, entry.reason[:140]]],
      headers=("record", "facet", "transfer said", "why the pool refuses"))

Had Gary filed its per-protocol number — the effect among the families who enrolled,
a latent subpopulation — the transfer would have *downgraded* it into a whole-population
effect under an assumption, and the pool refuses to average it anyway. **Pooling is not
transferring.** A transfer attaches a named assumption to one number and carries it in
the ledger. A pool takes a precision-weighted average whose weights know nothing about
populations, and the assumption would apply to a quantity that never appears in the
output.

## 5 · Does every party mean the same thing by hours?

The estimands agree on the outcome's name, unit and dimension. They do not know that
Gary re-registered what "hours worked" means in June 1973. The definition registry
does, and `commensurable` can be asked to resolve each record's party through it.

In [ ]:
catalog = Catalog(Path(tempfile.mkdtemp()) / "income-maintenance")
definitions = DefinitionRegistry(catalog)
for site in programme.sites:
    catalog.register(party_of[site.party])
    definitions.register(party_of[site.party], "hours_worked", W.hours_definition("self_report"), at=f"{site.started}-03-01")
definitions.register(party_of["indiana"], "hours_worked", W.hours_definition("employer_verified"), at="1973-06-01",
                     note="reconciled against employer records after the earnings audit")
consensus = definitions.consensus("hours_worked", list(party_of.values()))
print(consensus.summary())
print(" verdict:", consensus.verdict().status)

checked = commensurable(corpus, estimands, reference=reference,
                        definitions=definitions, programs=party_of, term="hours_worked")
print("\nthe pool, resolving definitions:", checked.verdict().status)
table([[e.study, e.facet, e.reason[:110]] for e in checked.entries],
      headers=("record", "facet", "why"), title="records the pool may not average with the reference")

Four records blocked — every Gary arm — because the party that filed them currently
means something different by the outcome than the party that filed the reference. The
estimands could not have said so; they carry the outcome's name, and the name did not
change. That is the failure the registry exists to catch, and it is a *block* rather
than an assumption because "self-reported hours are commensurable with
employer-verified hours" is not a thing anybody has agreed.

What a programme does about it is a decision: re-express Gary's records on the
self-report scale, or drop them, or get all four parties to register the same term.
Here the corpus is pooled **without** Gary and the difference recorded.

In [ ]:
without_gary = Corpus(records=tuple(r for r in corpus.records if r.contributor != "indiana"), name="NIT-4 without Gary")
party_index_wo = [list(party_of).index(r.contributor) for r in without_gary.records]
levels_wo = three_level([r.estimate for r in without_gary.records], [r.se for r in without_gary.records],
                        list(range(len(without_gary))), party_index_wo)
table([["all four parties", len(corpus), f"{levels.estimate:+.2%}", levels.interval.text(), f"{levels.tau_party:.2%}"],
       ["without Gary", len(without_gary), f"{levels_wo.estimate:+.2%}", levels_wo.interval.text(), f"{levels_wo.tau_party:.2%}"]],
      headers=("corpus", "records", "three-level mean", "95% interval", "tau (between parties)"))

## 6 · The decisions, taken together

Each experiment reported two things: the hours contrast, and a guardrail — the rate of
marital dissolution among plan families against controls, which Seattle-Denver's
analysts found elevated and reported. Eight readouts, eight go/no-go decisions, and the
question is not whether each is defensible on its own but how many of them a programme
that judged each at five per cent would get wrong.

One thing to say before the table, because the case study would be dishonest without
it: the synthetic world plants **no** dissolution effect anywhere, and the seed the
dissolution draws come from (`W.DISSOLUTION_SEED`) was chosen so that the Seattle-Denver
null draw lands at about $p = 0.02$ — a one-in-fifty draw, picked on purpose. A programme
that takes eight decisions a period meets a draw like that every few periods, and the
section is about what it does with one.

`program_decisions` refuses to choose the correction. It puts the uncorrected count
beside the corrected ones so that choosing `none` is a choice somebody made.

In [ ]:
readouts = []
for site in programme.sites:
    observed = programme[site.name].observed
    itt = intention_to_treat(observed, "hours_observed", "assigned")
    p_primary = float(2 * stats.norm.sf(abs(itt.estimate / itt.se)))
    f = programme[site.name].frame
    dissolved = pd.crosstab(f.assigned, f.dissolved)
    p_guardrail = float(stats.chi2_contingency(dissolved.to_numpy())[1])
    for metric, p in (("primary", p_primary), ("dissolution", p_guardrail)):
        readouts.append(Readout(experiment=site.experiment, party=site.party, metric=metric,
                                p_value=p, evalue=from_p_value(p)))
table([[r.key, f"{r.p_value:.2g}", f"{r.evalue:.3g}"] for r in readouts],
      headers=("readout", "p", "calibrated e-value"), title="eight decisions")
sd = programme["seattle_denver"].frame
print(f"\nSeattle-Denver dissolution: {sd.loc[sd.assigned == 1, 'dissolved'].mean():.1%} of plan families against "
      f"{sd.loc[sd.assigned == 0, 'dissolved'].mean():.1%} of controls — and the synthetic world's truth is no effect")

In [ ]:
rows = []
for method in ("none", "holm", "benjamini_hochberg", "e_bh"):
    report = program_decisions(readouts, alpha=0.05, method=method, period="1968-1982")
    rows.append([method, len(report.go), ", ".join(k.split("/")[1] for k in report.go if "dissolution" in k) or "—",
                 report.bound[:70] + "…"])
table(rows, headers=("method", "go", "guardrail 'findings' that survive", "what it controls"),
      title=f"eight readouts; a null programme expects {report.expected_false_go_uncorrected:.1f} false goes uncorrected")
show(program_decisions(readouts, alpha=0.05, method="e_bh", period="1968-1982").ledger_line())

Uncorrected, the dissolution finding is a go. Under Holm and under e-BH — the one
route that assumes nothing about how eight readouts from four sites in the same
economy depend on each other — it is not, and the four hours contrasts are. Which is
how the history went: the Seattle-Denver dissolution result was the finding that
reached Congress, and the finding that did not survive reanalysis.

## 7 · A prior for the fifth site

Nobody ran a fifth experiment. If somebody did, `prior_from_pool` hands the pooled
posterior forward as its prior — and with a party level in the pool there are two
different fifth sites: another Seattle-Denver cohort, which crosses one level, and a
contractor nobody has seen, which crosses both.

In [ ]:
rows = []
for target, party in (("mu", ""), ("predictive", ""), ("same_party", "sri"), ("new_party", "")):
    prior, line = prior_from_pool(nested, target=target, party=party)
    rows.append([target, party or "—", f"{prior.hyper['mu']:+.2%}", f"{prior.hyper['sigma']:.2%}"])
table(rows, headers=("target", "party", "prior mean", "prior sd"),
      title="the nested pool, handed forward as a prior on the relative change at the average plan")
same, _ = prior_from_pool(nested, target="same_party", party="sri")
fresh, _ = prior_from_pool(nested, target="new_party")
print(f"a new Seattle-Denver cohort: sd {same.hyper['sigma']:.2%}; a contractor nobody has seen: "
      f"sd {fresh.hyper['sigma']:.2%} ({fresh.hyper['sigma'] / same.hyper['sigma']:.1f}x wider)")

## What notebook 4 established

1. **Thirty-three records are four contractors.** The two-level pools' intervals on
   the family mean are two to three percentage points wide; the three-level and nested
   pools, which count parties, are eight, and the between-party spread they find is
   the synthetic world's site scale.
2. **The plan is a moderator and the pool can carry it.** The nested pool's coefficients
   on the guarantee and the tax rate are both negative and both clear of zero, with each
   record shrunk toward its own contractor's mean.
3. **A facet difference is an assumption, not a block** — until the population is
   latent. The pool averages three-year and five-year cohorts and New Jersey's moved
   reference under named assumptions on the result, and refuses a complier-population
   record outright, because a pool's weights cannot carry an assumption.
4. **Gary means something different by hours, and only the registry knew.** Four
   records blocked on the outcome facet through a definition the estimands never saw.
5. **One guardrail finding, and it does not survive the programme.** Uncorrected, the
   dissolution readout is a go; under Holm and e-BH it is not, and the four hours
   contrasts are. Eight decisions judged separately would have been wrong once.
6. **The fifth site's prior is wider than any of the four**, and it depends on who runs
   it — which is the party level earning its keep.